# Tech Challenge ? Fase 1 (IADT)

## Classifica??o de c?ncer de mama

Este ? o notebook principal da an?lise com **dados estruturados** do Breast Cancer Wisconsin. A proposta aqui ? apresentar, de forma clara e organizada, o fluxo obrigat?rio do trabalho: entender o problema, explorar a base, preparar os dados, treinar os modelos e interpretar os resultados.

Ao longo do notebook, seguimos este percurso:

- contexto do problema e da base escolhida
- carregamento e explora??o dos dados
- pr?-processamento
- treinamento de diferentes modelos
- avalia??o com accuracy, recall, F1-score, ROC e matrizes de confus?o
- explicabilidade
- ajuste de limiar com foco cl?nico

Os estudos complementares com radi?mica, CNN e multimodalidade ficam separados no notebook `02_estudo_integrado.ipynb`, para preservar aqui o n?cleo principal da entrega.


## 1. Prepara??o do ambiente

Nesta etapa, adicionamos o diret?rio `src/` ao caminho de busca do Python. Com isso, o pacote do projeto pode ser importado diretamente no notebook, sem exigir uma instala??o pr?via no ambiente.


In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import cross_validate

from techchallenge import config
from techchallenge.data import estruturado as dados
from techchallenge.models import estruturado as modelos_estr
from techchallenge.evaluation import metrics, explicabilidade, limiar

sns.set_theme(style="whitegrid")
OUT = config.garantir_outputs()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
config.PROJETO_DIR, OUT

## 2. Problema e base escolhida

O objetivo ? classificar tumores de mama em **malignos** ou **benignos** a partir de caracter?sticas num?ricas extra?das dos exames. Para essa etapa do trabalho, utilizamos o **Breast Cancer Wisconsin Dataset**, uma base bastante conhecida em tarefas de classifica??o supervisionada, com 569 registros e 30 vari?veis preditoras.

Antes da an?lise, a fun??o `carregar_dados()` faz uma limpeza inicial na base. Nesse processo, as colunas `id` e `Unnamed: 32` s?o removidas: a primeira ? apenas um identificador de registro, enquanto a segunda est? vazia e n?o acrescenta informa??o ?til ao modelo.


In [ ]:
df = dados.carregar_dados()
print("Formato:", df.shape)
df.head()

## 3. An?lise explorat?ria dos dados (EDA)

Antes de treinar os modelos, vale entender como a base se comporta. Nesta etapa, observamos a distribui??o da vari?vel-alvo, as estat?sticas descritivas e as correla??es mais associadas ? malignidade.

Essa leitura inicial ajuda a contextualizar o problema, identificar poss?veis padr?es e levantar hip?teses sobre quais vari?veis tendem a ter maior peso na classifica??o.


In [ ]:
counts = df["diagnosis"].value_counts().rename(index={"B": "Benigno", "M": "Maligno"})
display(counts.to_frame("quantidade"))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="diagnosis", order=["B", "M"], palette=["#4c9f70", "#d1495b"])
plt.title("Distribuicao do diagnostico")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
df.describe().T.head(10)

In [ ]:
df_corr = df.copy()
df_corr["diagnosis"] = df_corr["diagnosis"].map({"M": 1, "B": 0})
corr = df_corr.corr(numeric_only=True)["diagnosis"].drop("diagnosis").sort_values(ascending=False)
print("Top 10 features mais correlacionadas com malignidade:")
display(corr.head(10).round(3).to_frame("correlacao"))

plt.figure(figsize=(8, 10))
corr.head(15).sort_values().plot(kind="barh", color="#d1495b")
plt.title("Top correlacoes com malignidade")
plt.xlabel("Correlacao de Pearson")
plt.show()

In [ ]:
top_cols = corr.head(10).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df_corr[top_cols + ["diagnosis"]].corr(), cmap="coolwarm", center=0)
plt.title("Heatmap das principais features")
plt.show()

## 4. Pr?-processamento

Como a base possui mais casos benignos do que malignos, utilizamos uma divis?o **estratificada** entre treino e teste. Isso garante que a propor??o entre as classes seja preservada nos dois conjuntos e torna a avalia??o mais representativa.

Al?m disso, o pacote do projeto aplica internamente as seguintes etapas:

- remo??o de colunas n?o informativas
- codifica??o da vari?vel-alvo (`M = 1` e `B = 0`)
- separa??o entre treino e teste com estratifica??o
- padroniza??o dentro de `Pipeline`, evitando vazamento de dados

Esse cuidado ? importante para manter o fluxo consistente entre os modelos e assegurar que a compara??o seja justa.


In [ ]:
X_train, X_test, y_train, y_test = dados.preparar_treino_teste(df)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporcao no treino:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporcao no teste :", y_test.value_counts(normalize=True).round(3).to_dict())

## 5. Modelagem

Nesta etapa, treinamos os **seis modelos de base** do projeto: Regress?o Log?stica, ?rvore de Decis?o, KNN, Random Forest, Gradient Boosting e SVM. Cada modelo ? organizado dentro de um *pipeline*, o que mant?m o pr?-processamento acoplado ao classificador e reduz o risco de inconsist?ncias entre treino e teste.

Al?m dos modelos individuais, avaliamos tamb?m uma estrat?gia de **Stacking**. Nessa abordagem, os modelos de base geram previs?es no primeiro n?vel, e um metamodelo de Regress?o Log?stica aprende a combinar esses sinais para produzir a decis?o final.

Na pr?tica, essa combina??o busca aproveitar for?as diferentes de cada t?cnica. Para evitar vazamento de informa??o, o Stacking utiliza valida??o cruzada interna antes do ajuste final do metamodelo.


In [ ]:
modelos = modelos_estr.construir_modelos()
modelos["Stacking"] = modelos_estr.construir_stacking(modelos_estr.construir_modelos())

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    linha = {"Modelo": nome, **metrics.metricas(y_test, y_pred)}
    cv = cross_validate(modelo, X_train, y_train, cv=5, scoring=["accuracy", "recall"], n_jobs=-1)
    linha["CV Acc (media)"] = cv["test_accuracy"].mean()
    linha["CV Recall (media)"] = cv["test_recall"].mean()
    resultados.append(linha)

res_df = pd.DataFrame(resultados).set_index("Modelo").round(4).sort_values(
    by=["Recall (maligno)", "Accuracy"], ascending=False
)
res_df

## 6. Avalia??o

Na compara??o entre os modelos, a m?trica de maior interesse ? o **recall da classe maligna**, porque o erro mais cr?tico neste contexto ? o falso negativo ? ou seja, quando um caso maligno ? classificado como benigno.

Ainda assim, o recall n?o ? analisado isoladamente. Tamb?m consideramos **accuracy**, **F1-score**, **ROC/AUC** e as **matrizes de confus?o**, para manter uma vis?o mais equilibrada do desempenho de cada abordagem. Todos os modelos s?o avaliados sobre o mesmo conjunto de teste.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for ax, (nome, modelo) in zip(axes, modelos.items()):
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Benigno", "Maligno"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)

for ax in axes[len(modelos):]:
    ax.axis("off")

plt.suptitle("Matrizes de confusao - modelos do trabalho", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
for nome, modelo in modelos.items():
    if hasattr(modelo, "predict_proba"):
        prob = modelo.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        lw = 2.5 if nome == "Stacking" else 1.3
        plt.plot(fpr, tpr, lw=lw, label=f"{nome} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("Falso positivo (1 - especificidade)")
plt.ylabel("Verdadeiro positivo (recall)")
plt.title("Curvas ROC - modulo estruturado")
plt.legend(loc="lower right", fontsize=8)
plt.show()

## 7. Explicabilidade

Como o Stacking combina m?ltiplos modelos, sua interpreta??o direta se torna menos transparente. Por isso, complementamos a an?lise com duas vis?es mais explic?veis: os coeficientes da Regress?o Log?stica e os valores SHAP da ?rvore de Decis?o.

Esses recursos ajudam a identificar quais caracter?sticas mais influenciam as previs?es e tornam a discuss?o dos resultados mais s?lida do ponto de vista anal?tico.


In [ ]:
explicabilidade.feature_importance_logistica(
    modelos["Regressao Logistica"],
    X_train.columns,
    OUT / "estruturado_feature_importance.png",
)
display(Image(str(OUT / "estruturado_feature_importance.png")))

In [ ]:
explicabilidade.shap_arvore(
    modelos["Arvore de Decisao"],
    X_test,
    X_train.columns,
    OUT / "estruturado_shap.png",
)
display(Image(str(OUT / "estruturado_shap.png")))

## 8. Ajuste de limiar com prioridade cl?nica

Depois de identificar o melhor modelo global, analisamos o efeito de variar o limiar de decis?o. Esse passo ? importante porque, em cen?rios cl?nicos, muitas vezes faz sentido aceitar mais falsos positivos para reduzir a chance de falsos negativos.

Aqui, o foco ? entender esse equil?brio e verificar em que ponto o modelo passa a favorecer mais o **recall** da classe maligna.


In [ ]:
prob = modelos["Stacking"].predict_proba(X_test)[:, 1]
tab = limiar.tabela_limiares(y_test, prob)
t_recall = limiar.escolher_limiar_por_recall(y_test, prob, recall_alvo=1.0)
limiar.plot_trade_off(tab, OUT / "estruturado_limiar_tradeoff.png", t_recall)
limiar.comparar_matrizes(y_test, prob, 0.5, t_recall, OUT / "estruturado_limiar_matrizes.png")

def resumo_limiar(t):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    rec = tp / (tp + fn)
    return {"limiar": t, "FN": int(fn), "FP": int(fp), "Recall": round(rec, 4)}

pd.DataFrame([resumo_limiar(0.5), resumo_limiar(t_recall)])

In [ ]:
display(Image(str(OUT / "estruturado_limiar_tradeoff.png")))
display(Image(str(OUT / "estruturado_limiar_matrizes.png")))

## 9. Conclus?o

Este notebook consolida a parte principal do trabalho com dados estruturados e mostra, de ponta a ponta, como o problema foi tratado: da leitura da base at? a interpreta??o dos resultados.

De forma geral, o **Stacking** aparece como a alternativa mais forte em desempenho global, enquanto a **Regress?o Log?stica** continua sendo uma refer?ncia importante em interpretabilidade. Em um contexto aplicado, esse equil?brio entre desempenho e capacidade de explica??o ? especialmente valioso.

Os estudos complementares ? especialmente os relacionados a imagens, radi?mica e estrat?gias multimodais ? ficam organizados separadamente no notebook `02_estudo_integrado.ipynb`.
